# 04. Evaluar y visualizar modelos

Este notebook compara las predicciones locales generadas por el notebook 03 contra los `reference_claim` anotados de `test.csv`. METEOR, ROUGE-L y BERTScore miden similitud con la referencia; no validan la veracidad médica de una noticia.

Tiene dos partes:

- **Parte A. Evaluar una variante.** Se ejecuta una vez por variante, cambiando `VARIANT`. Guarda sus métricas, tablas y figuras dentro de la carpeta de la corrida, en `artifacts/03_corridas/`.
- **Parte B. Resultados finales.** Se ejecuta al final, cuando las cuatro variantes están evaluadas. Toma cada modelo con la variante elegida por validación y guarda en `artifacts/05_resultados_finales/` las tablas y figuras de la tesis.

## Requisitos

Ejecuta antes el notebook 03. Para METEOR y BERTScore, instala `pip install -e '.[analysis,models,dev]'` y descarga una vez los recursos de NLTK con `python -m nltk.downloader wordnet omw-1.4`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

DATASET_NAME = 'claims_1500'  # debe coincidir con el notebook 03
FINAL_DIR = ROOT / 'artifacts' / '02_conjuntos' / DATASET_NAME
from claim_extraction.workflow import BASE_SEED, DATASET_SPLIT_SIZES, EXPERIMENT_VARIANTS, run_name_for

VARIANT = 'epocas_10'  # debe coincidir con el notebook 03
RUN_NAME = run_name_for(DATASET_NAME, VARIANT)
RUNS_DIR = ROOT / 'artifacts' / '03_corridas'
RUN_DIR = RUNS_DIR / RUN_NAME
PREDICTIONS_DIR = RUN_DIR / 'predictions'
EVALUATION_DIR = RUN_DIR / 'evaluation'
TABLES_DIR = EVALUATION_DIR / 'tables'
FIGURES_DIR = EVALUATION_DIR / 'figures'
# Parte B: cada modelo con la variante elegida por validación.
FINAL_RESULTS_DIR = ROOT / 'artifacts' / '05_resultados_finales' / DATASET_NAME
EXPECTED_MODEL_IDS = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']
EXPECTED_SPLIT_SIZES = DATASET_SPLIT_SIZES[DATASET_NAME]

print({'dataset': DATASET_NAME, 'variant': VARIANT, 'run_dir': str(RUN_DIR), 'final_results_dir': str(FINAL_RESULTS_DIR)})

In [ ]:
import json

import pandas as pd

from claim_extraction.workflow import assert_run_matches, evaluate_predictions, split_fingerprints, utc_now, validate_sem1_splits

SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
splits = validate_sem1_splits(SPLIT_PATHS, expected_sizes=EXPECTED_SPLIT_SIZES)
test = splits['test']

manifest_path = RUN_DIR / 'run_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Falta el manifiesto de la corrida: {manifest_path}')
run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert_run_matches(run_manifest, dataset_name=DATASET_NAME, variant=VARIANT, fingerprints=split_fingerprints(splits), seed=BASE_SEED)
run_models = run_manifest.get('models', {})
if not isinstance(run_models, dict) or not run_models:
    raise ValueError('El manifiesto no registra modelos para evaluar.')
MODEL_ORDER = [model_id for model_id in EXPECTED_MODEL_IDS if model_id in run_models]
if not MODEL_ORDER:
    raise ValueError('El manifiesto no contiene IDs de modelos reconocidos.')

# Se verifica que cada modelo de esta corrida cubra exactamente las noticias de test.
prediction_paths = [PREDICTIONS_DIR / f'{model_id}.csv' for model_id in MODEL_ORDER]
missing = [path for path in prediction_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Faltan predicciones: ' + ', '.join(str(path) for path in missing))

prediction_frames = [pd.read_csv(path, dtype=str, keep_default_na=False) for path in prediction_paths]
predictions = pd.concat(prediction_frames, ignore_index=True)
expected_test_ids = set(test['record_id'])
for model_id, group in predictions.groupby('model_id', sort=True):
    if set(group['record_id']) != expected_test_ids or len(group) != len(test):
        raise ValueError(f'{model_id} no cubre exactamente los casos de test.csv.')

print({'test_rows': len(test), 'prediction_rows': len(predictions), 'models': sorted(predictions['model_id'].unique())})

## Parte A. Evaluar esta variante

### Calcular métricas

`summary.csv` guarda el promedio de cada modelo y `per_prediction.csv` el puntaje de cada noticia. Ambos quedan en la carpeta de la corrida. Los cuerpos de noticias, claims de referencia y predicciones individuales permanecen ignorados por Git.

In [ ]:
# Estas métricas comparan texto generado y referencia; no determinan veracidad médica.
summary, per_prediction = evaluate_predictions(predictions, test)
summary = summary.set_index('model_id').reindex(MODEL_ORDER).reset_index()
if summary[['meteor', 'rouge_l_f1', 'bertscore_f1']].isna().any().any():
    raise AssertionError('No se calcularon las tres métricas de referencia para todos los modelos.')

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(EVALUATION_DIR / 'summary.csv', index=False, encoding='utf-8')
per_prediction.to_csv(EVALUATION_DIR / 'per_prediction.csv', index=False, encoding='utf-8')

status_path = RUN_DIR / 'run_status.json'
status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
status['status'] = 'completed'
status['evaluated_at_utc'] = utc_now()
status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')

summary[['model_display_name', 'prediction_n', 'reference_n', 'meteor', 'rouge_l_f1', 'bertscore_f1']]

### Métricas y diagnósticos

La tabla de diagnósticos describe las salidas generadas: vacías, muy cortas o repetidas. Las mismas funciones se reutilizan en la parte B.

In [ ]:
import matplotlib.pyplot as plt

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRIC_LABELS = {'meteor': 'METEOR', 'rouge_l_f1': 'ROUGE-L F1', 'bertscore_f1': 'BERTScore F1'}


def metrics_table_from(model_summary):
    return model_summary[['model_id', 'model_display_name', 'prediction_n', 'reference_n', *METRIC_LABELS]].copy()


def diagnostics_table_from(model_summary):
    # Describe las salidas generadas; no mide veracidad médica.
    table = model_summary[[
        'model_display_name', 'prediction_n', 'empty_rate',
        'shorter_than_5_tokens_rate', 'mean_prediction_tokens_operational',
        'duplicate_output_rate',
    ]].copy().rename(columns={
        'model_display_name': 'Modelo',
        'prediction_n': 'Afirmaciones generadas (n)',
        'empty_rate': 'Salidas vacías (%)',
        'shorter_than_5_tokens_rate': 'Menos de 5 tokens (%)',
        'mean_prediction_tokens_operational': 'Longitud media (tokens léxicos)',
        'duplicate_output_rate': 'Repeticiones adicionales (%)',
    })
    for column in ('Salidas vacías (%)', 'Menos de 5 tokens (%)', 'Repeticiones adicionales (%)'):
        table[column] = (100 * table[column]).round(1)
    table['Longitud media (tokens léxicos)'] = table['Longitud media (tokens léxicos)'].round(2)
    return table


def plot_metrics(metrics_table, path):
    figure, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout='constrained')
    for axis, (column, label) in zip(axes, METRIC_LABELS.items(), strict=True):
        bars = axis.bar(metrics_table['model_display_name'], metrics_table[column], color='#277f8e')
        axis.bar_label(bars, labels=[f'{value:.3f}' for value in metrics_table[column]], padding=3, fontsize=9)
        axis.set(title=label, ylabel='Puntaje medio', ylim=(0, 1))
        axis.tick_params(axis='x', rotation=20)
        axis.grid(axis='y', alpha=.2)
    figure.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()


metrics_table = metrics_table_from(summary)
diagnostics_table = diagnostics_table_from(summary)
metrics_table.to_csv(TABLES_DIR / '01_metricas_principales.csv', index=False, encoding='utf-8')
diagnostics_table.to_csv(TABLES_DIR / '05_diagnosticos_generacion.csv', index=False, encoding='utf-8')
plot_metrics(metrics_table, FIGURES_DIR / '01_comparacion_metricas.png')

print(f'Resultados de la variante: {EVALUATION_DIR.relative_to(ROOT)}')
diagnostics_table

### Curvas de entrenamiento y validación

Se leen los historiales producidos por `Seq2SeqTrainer`. La línea vertical marca el checkpoint seleccionado por menor `eval_loss` dentro de cada arquitectura; no compara pérdidas entre arquitecturas.

In [ ]:
from claim_extraction.workflow import training_history


def plot_training_curves(model_dirs, display_names, path):
    """Dibuja la pérdida de entrenamiento y validación de cada modelo y marca la época elegida."""
    figure, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
    for axis in axes.flat[len(model_dirs):]:
        axis.set_visible(False)  # una corrida puede incluir menos de cuatro modelos
    chosen = []
    for axis, (model_id, model_dir) in zip(axes.flat, model_dirs.items()):
        history = training_history(model_dir).get('log_history', [])
        training = [(row['epoch'], row['loss']) for row in history if 'loss' in row and 'epoch' in row]
        validation = [(row['epoch'], row['eval_loss']) for row in history if 'eval_loss' in row and 'epoch' in row]
        if not training or not validation:
            raise AssertionError(f'Faltan pérdidas de entrenamiento o validación para {model_id}.')
        best_epoch = min(validation, key=lambda point: point[1])[0]
        axis.plot(*zip(*training), label='Entrenamiento', color='#245b78')
        axis.plot(*zip(*validation), label='Validación', marker='o', color='#c77c35')
        axis.axvline(best_epoch, color='#777777', linestyle=':', label=f'Checkpoint elegido (época {best_epoch:g})')
        axis.set(title=display_names[model_id], xlabel='Época', ylabel='Pérdida')
        axis.grid(alpha=.2)
        axis.legend(fontsize=8, frameon=False)
        chosen.append({'model_id': model_id, 'checkpoint_epoch': best_epoch})
    figure.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    return pd.DataFrame(chosen)


plot_training_curves(
    {model_id: RUN_DIR / 'models' / model_id for model_id in MODEL_ORDER},
    {model_id: run_models[model_id]['display_name'] for model_id in MODEL_ORDER},
    FIGURES_DIR / '02_curvas_entrenamiento_validacion.png',
)

### Incertidumbre, sensibilidad y resultados por tema

Las medias de test llevan un intervalo de confianza del 95 % por bootstrap sobre las noticias. La diferencia entre modelos se calcula de forma pareada contra el de mayor media: si su intervalo incluye el cero, esos dos modelos no se distinguen con este test.

METEOR se calcula como en NLTK por defecto, con stemmer y WordNet en inglés. `meteor_stem_es` repite el cálculo con stemmer español como análisis de sensibilidad; no reemplaza a la métrica principal.

La tabla por tema usa el `tema` anotado. Con grupos pequeños los intervalos son amplios: revisa siempre `n` antes de comparar temas.

In [ ]:
from claim_extraction.analysis import bootstrap_uncertainty, metrics_by_topic, with_spanish_meteor

STAT_METRICS = ('meteor', 'meteor_stem_es', 'rouge_l_f1', 'bertscore_f1')


def confidence_tables(model_per_prediction):
    evaluated = with_spanish_meteor(model_per_prediction)
    intervals, differences = [], []
    for metric in STAT_METRICS:
        metric_intervals, metric_differences = bootstrap_uncertainty(evaluated, metric=metric)
        intervals.append(metric_intervals)
        differences.append(metric_differences)
    return evaluated, pd.concat(intervals, ignore_index=True), pd.concat(differences, ignore_index=True)


evaluated, uncertainty, differences = confidence_tables(per_prediction)
uncertainty.to_csv(TABLES_DIR / '02_intervalos_confianza.csv', index=False, encoding='utf-8')
differences.to_csv(TABLES_DIR / '03_diferencias_pareadas.csv', index=False, encoding='utf-8')

if 'tema' in test:
    by_topic = metrics_by_topic(per_prediction, test)
    by_topic.to_csv(TABLES_DIR / '04_metricas_por_tema.csv', index=False, encoding='utf-8')
    display(by_topic)
else:
    print('Este conjunto no tiene tema; se omite la tabla por tema.')
display(uncertainty)
differences

### Comparar variantes

Reúne las variantes ya evaluadas. La columna `elegida_por_validacion` marca, para cada modelo, la variante con menor pérdida de validación. Esa elección no usa el test: METEOR, ROUGE-L y BERTScore solo se reportan.

In [ ]:
# Evalúa cada variante cambiando VARIANT antes de ejecutar esta comparación.
comparison_rows = []
for variant in EXPERIMENT_VARIANTS:
    variant_dir = RUNS_DIR / run_name_for(DATASET_NAME, variant)
    summary_path = variant_dir / 'evaluation' / 'summary.csv'
    if not summary_path.is_file():
        continue
    variant_summary = pd.read_csv(summary_path, encoding='utf-8').set_index('model_id')
    for model_id in EXPECTED_MODEL_IDS:
        if model_id not in variant_summary.index:
            continue
        evaluations = [row for row in training_history(variant_dir / 'models' / model_id)['log_history'] if 'eval_loss' in row]
        best = min(evaluations, key=lambda row: row['eval_loss'])
        metrics = variant_summary.loc[model_id]
        comparison_rows.append({
            'modelo': metrics['model_display_name'], 'model_id': model_id, 'variante': variant,
            'epocas_ejecutadas': round(evaluations[-1]['epoch']), 'mejor_epoca': round(best['epoch']),
            'perdida_validacion': best['eval_loss'], 'meteor_test': metrics['meteor'],
            'rouge_l_f1_test': metrics['rouge_l_f1'], 'bertscore_f1_test': metrics['bertscore_f1'],
        })
if not comparison_rows:
    raise RuntimeError('Todavía no hay variantes evaluadas.')

comparison = pd.DataFrame(comparison_rows)
comparison['elegida_por_validacion'] = comparison.index.isin(comparison.groupby('model_id')['perdida_validacion'].idxmin())
comparison.to_csv(RUNS_DIR / f'comparacion_variantes_{DATASET_NAME}.csv', index=False, encoding='utf-8')
comparison

## Parte B. Resultados finales

Ejecuta esta parte al final, cuando las cuatro variantes de los cuatro modelos estén entrenadas y evaluadas; si falta alguna, el notebook se detiene aquí y lo indica. Cada modelo entra con la variante que eligió la validación, y todo se guarda en `artifacts/05_resultados_finales/`: es la carpeta que se revisa para redactar la tesis.

Las reglas están fijadas en `docs/EXPERIMENT_PROTOCOL.md`, sección 4.2. Estos puntajes comparan similitud con la referencia; no validan la veracidad médica.

In [ ]:
from claim_extraction.workflow import load_selected_evaluations

missing_variants = {
    model_id: sorted(set(EXPERIMENT_VARIANTS) - set(comparison.loc[comparison['model_id'].eq(model_id), 'variante']))
    for model_id in EXPECTED_MODEL_IDS
}
missing_variants = {model_id: variants for model_id, variants in missing_variants.items() if variants}
if missing_variants:
    raise RuntimeError(f'Faltan variantes por evaluar antes de los resultados finales: {missing_variants}')

selected = (
    comparison[comparison['elegida_por_validacion']]
    .drop(columns='elegida_por_validacion')
    .set_index('model_id').reindex(EXPECTED_MODEL_IDS).reset_index()
)
SELECTED_VARIANTS = dict(zip(selected['model_id'], selected['variante'], strict=True))
final_summary, final_per_prediction = load_selected_evaluations(
    RUNS_DIR,
    dataset_name=DATASET_NAME,
    selected_variants=SELECTED_VARIANTS,
    fingerprints=split_fingerprints(splits),
)
final_summary = final_summary.set_index('model_id').reindex(EXPECTED_MODEL_IDS).reset_index()
final_names = dict(zip(final_summary['model_id'], final_summary['model_display_name'], strict=True))

FINAL_TABLES_DIR = FINAL_RESULTS_DIR / 'tablas'
FINAL_FIGURES_DIR = FINAL_RESULTS_DIR / 'figuras'
FINAL_TABLES_DIR.mkdir(parents=True, exist_ok=True)
FINAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

split_table = pd.DataFrame([{'split': name, 'filas': len(frame)} for name, frame in splits.items()])
final_evaluated, final_intervals, _ = confidence_tables(final_per_prediction)
final_diagnostics = diagnostics_table_from(final_summary)
final_diagnostics.insert(1, 'Variante', final_summary['variante'].to_numpy())

split_table.to_csv(FINAL_TABLES_DIR / '01_distribucion_particiones.csv', index=False, encoding='utf-8')
selected.to_csv(FINAL_TABLES_DIR / '02_variantes_elegidas.csv', index=False, encoding='utf-8')
final_intervals.to_csv(FINAL_TABLES_DIR / '03_metricas_finales.csv', index=False, encoding='utf-8')
final_diagnostics.to_csv(FINAL_TABLES_DIR / '07_diagnosticos_generacion.csv', index=False, encoding='utf-8')

plot_metrics(metrics_table_from(final_summary), FINAL_FIGURES_DIR / '01_comparacion_metricas.png')
plot_training_curves(
    {model_id: RUNS_DIR / run_name_for(DATASET_NAME, variant) / 'models' / model_id for model_id, variant in SELECTED_VARIANTS.items()},
    final_names,
    FINAL_FIGURES_DIR / '02_curvas_entrenamiento_validacion.png',
)
display(selected)
final_intervals

### Pruebas estadísticas entre modelos

- **Friedman** compara los cuatro modelos a la vez sobre las mismas noticias.
- **Wilcoxon** compara cada par de modelos noticia por noticia, y **Holm** corrige las seis comparaciones.
- **Lectura:** la columna `lectura` da la frase que permite cada resultado. Un par solo se lee como diferencia si Friedman la detecta primero. Que dos modelos no se distingan no significa que sean iguales.

METEOR es la métrica principal. Las demás son complementarias y cada una se corrige por separado.

In [ ]:
from claim_extraction.analysis import model_significance_tests

friedman_rows, pair_tables = [], []
for metric in STAT_METRICS:
    friedman, pairs = model_significance_tests(final_evaluated, metric=metric)
    friedman_rows.append(friedman)
    pair_tables.append(pairs)
friedman_table = pd.DataFrame(friedman_rows)
pairwise_tests = pd.concat(pair_tables, ignore_index=True)
friedman_table.to_csv(FINAL_TABLES_DIR / '04_friedman.csv', index=False, encoding='utf-8')
pairwise_tests.to_csv(FINAL_TABLES_DIR / '05_comparaciones_pareadas.csv', index=False, encoding='utf-8')

if 'tema' in test:
    final_by_topic = metrics_by_topic(final_per_prediction, test)
    final_by_topic.to_csv(FINAL_TABLES_DIR / '06_metricas_por_tema.csv', index=False, encoding='utf-8')
else:
    print('Este conjunto no tiene tema; se omite la tabla por tema.')

display(friedman_table)
pairwise_tests[pairwise_tests['metric'].eq('meteor')]

### Distribución de longitudes

Describe el conjunto, no una variante, por eso se calcula una sola vez. La longitud de noticia combina `title` y `body`; la de referencia corresponde al `reference_claim` anotado.

In [ ]:
# Esta figura describe el tamaño del material de entrada y de las afirmaciones, no la calidad del modelo.
import re

all_records = pd.concat([splits[name] for name in ('train', 'validation', 'test')], ignore_index=True)
word_count = lambda value: len(re.findall(r'\b\w+\b', str(value), flags=re.UNICODE))
article_text = all_records['title'].fillna('') + ' ' + all_records['body'].fillna('')
lengths = pd.DataFrame({
    'news_words_title_body': article_text.map(word_count),
    'reference_claim_words': all_records['reference_claim'].fillna('').map(word_count),
})
if len(lengths) != sum(EXPECTED_SPLIT_SIZES.values()):
    raise AssertionError('La distribución debe incluir exactamente los registros del conjunto final.')

figure, axes = plt.subplots(2, 1, figsize=(8, 7), layout='constrained')
for axis, column, title, color in [
    (axes[0], 'news_words_title_body', 'Noticias (título y cuerpo)', '#245b78'),
    (axes[1], 'reference_claim_words', 'Afirmaciones de referencia', '#27877c'),
]:
    median = lengths[column].median()
    axis.hist(lengths[column], bins=20, color=color, edgecolor='white')
    axis.axvline(median, color='#b75a3c', linestyle='--', label=f'Mediana: {median:.0f}')
    axis.set(title=title, xlabel='Longitud (palabras)', ylabel='Frecuencia')
    axis.legend(frameon=False)
    axis.grid(axis='y', alpha=.2)
figure.savefig(FINAL_FIGURES_DIR / '03_distribucion_longitudes.png', dpi=180, bbox_inches='tight')
plt.show()

lengths.describe().T[['count', 'mean', '50%', 'min', 'max']]

### Cantidad de tokens

Cada modelo divide el texto con su propio tokenizador, así que el mismo texto ocupa una cantidad distinta de tokens en cada uno. Las figuras cuentan los tokens sin truncar sobre todos los registros del conjunto final y marcan el límite usado en el entrenamiento: lo que lo supera no llega al modelo.

In [ ]:
# Se cuenta sin truncar para ver cuánto texto queda fuera del límite de entrenamiento.
from claim_extraction.models import load_tokenizer
from claim_extraction.registry import get_model
from claim_extraction.text import input_for_model

input_texts = [input_for_model(row) for row in all_records.to_dict(orient='records')]
target_texts = all_records['reference_claim'].map(str).tolist()

token_rows = []
token_limits = {}
for model_id in EXPECTED_MODEL_IDS:
    spec = get_model(model_id)
    tokenizer = load_tokenizer(spec.base_checkpoint, model_id=model_id)
    token_limits[model_id] = {
        'source_tokens': int(spec.training['max_source_length']),
        'target_tokens': int(spec.training['max_target_length']),
    }
    source_ids = tokenizer(input_texts, truncation=False, verbose=False)['input_ids']
    target_ids = tokenizer(text_target=target_texts, truncation=False, verbose=False)['input_ids']
    for source, target in zip(source_ids, target_ids, strict=True):
        token_rows.append({'model_id': model_id, 'source_tokens': len(source), 'target_tokens': len(target)})
token_counts = pd.DataFrame(token_rows)


def plot_token_histograms(column, title, color, filename):
    figure, axes = plt.subplots(2, 2, figsize=(12, 7), layout='constrained', sharex=True, sharey=True)
    for axis, model_id in zip(axes.flat, EXPECTED_MODEL_IDS, strict=True):
        counts = token_counts.loc[token_counts['model_id'].eq(model_id), column]
        limit = token_limits[model_id][column]
        axis.hist(counts, bins=30, color=color, edgecolor='white')
        axis.axvline(counts.median(), color='#777777', linestyle=':', label=f'Mediana: {counts.median():.0f}')
        axis.axvline(limit, color='#b75a3c', linestyle='--', label=f'Límite: {limit} ({counts.gt(limit).mean():.1%} lo supera)')
        axis.set(title=final_names[model_id], xlabel='Tokens', ylabel='Noticias')
        axis.grid(axis='y', alpha=.2)
        axis.legend(fontsize=9, frameon=False)
    figure.suptitle(title)
    figure.savefig(FINAL_FIGURES_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()


plot_token_histograms('source_tokens', 'Tokens de entrada (título y cuerpo)', '#245b78', '04_tokens_noticias.png')
plot_token_histograms('target_tokens', 'Tokens de las afirmaciones de referencia', '#27877c', '05_tokens_afirmaciones.png')

token_summary = pd.DataFrame([
    {
        'Modelo': final_names[model_id],
        'Texto': label,
        'Mediana': group[column].median(),
        'Percentil 95': group[column].quantile(.95),
        'Máximo': group[column].max(),
        'Límite': token_limits[model_id][column],
        'Superan el límite (%)': round(100 * group[column].gt(token_limits[model_id][column]).mean(), 1),
    }
    for model_id, group in token_counts.groupby('model_id', sort=False)
    for column, label in [('source_tokens', 'Noticia'), ('target_tokens', 'Afirmación')]
])
token_summary.to_csv(FINAL_TABLES_DIR / '08_resumen_tokens.csv', index=False, encoding='utf-8')
print(f'Resultados finales: {FINAL_RESULTS_DIR.relative_to(ROOT)}')
token_summary